# GPT Responses API Lab

In this lab we build a small chat assistant with OpenAI's Responses API, then add the features you would use on real analytics work.

Objectives:
- Call a GPT model and read the response, including token usage
- Control behavior with instructions, verbosity, and reasoning effort
- Stream replies and keep a conversation going
- Get structured data back instead of free text
- Let the model search the web and cite its sources
- Build a simple Gradio chatbot

## Game Plan
- **Context:** we run in Google Colab. `openai` and `gradio` come preinstalled, so there is nothing to install.
- **Secret management:** we read the API key from the Colab secret named `OpenAI_API_Key`.
- **Model:** `gpt-6-luna`, OpenAI's cheapest current model.

| Model | Good for | Input / Output per 1M tokens |
|---|---|---|
| `gpt-6-luna` | High-volume, focused tasks | \$0.10 / \$0.50 |
| `gpt-6.1-terra` | Visual reasoning, clear writing | \$1 / \$5 |
| `gpt-6.1-sol` | Balance of intelligence and cost | \$2 / \$10 |
| `gpt-6-astra` | Hardest reasoning and coding | \$10 / \$50 |

Prices are from OpenAI's pricing page in November 2026. Check [the current list](https://developers.openai.com/api/docs/pricing) before you plan a big job.

In [18]:
import os
from typing import Literal

import gradio as gr
import pandas as pd
from google.colab import userdata
from IPython.display import Markdown
from openai import OpenAI
from pydantic import BaseModel

MODEL = "gpt-6-luna"

## Load Secrets (No Hard-Coding!)
Colab keeps keys in its Secrets panel (the key icon on the left). Add `OpenAI_API_Key` there and turn on notebook access. Never paste a key into a cell.

In [19]:
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI_API_Key")
client = OpenAI()

## Your First Response
One call to `client.responses.create()`. The result holds the reply plus metadata such as token usage, which is what you pay for.

In [20]:
response = client.responses.create(
    model=MODEL,
    input="Write a one-sentence bedtime story about a unicorn."
)

response

Response(id='resp_0740677b843f548a006aee050e6bfc87d18769b5cd330ebf58', created_at=1793983758.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseReasoningItem(id='rs_0740677b843f548a006aee050f8b5887d1b1a316405b0db4d7', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq7gUQ_zkvYTz-CLO2vsWrxf-2HCABc1lYTsCGX4t1Bm7xQGYZQtaPgjKJdtnEqXX_EEUlRH-fv9c9P4Z21oZt572kmFU9dlMeofBzGIZcS1fQD5X86FnKFLCoeRgBUxBul8Yg2t8QPUAyS38xng749QZgO8gSxvNpruvU7b3MCoBL09GEDCCbdUsH4xAgE3zIvFtHySp8M16DFJB4bEEvp0ZDWDpsRNVGOy8F8J_t8bZeP-2gC_lKYH299l7UVmDiWfanBeSIXovO03L8OK-oa8Pft-_u5ArPQp9d8LXU11XmlugoX4-QHv29HoF81RZhXMtkU5zyPac3RkjUBj1_bq-UmEQIlW0kG005O48cc1ZO2jHkPQqhk1RVMaGPuir-GnLZCanUBy02eot4G9Q7Vyw14CSN2fx18cd18GnnQCjtHUN1iu1BkvxoXNr7W6oy8MEw-5WgQDBMFANQ4WIYrJsNTsM0AExHC6qtuNfAz9HZdIwoJHdC5j3lE3BUiWLLpvYwhYeQ1B60CjcIdL5RJRgLQYwKQgh9YT2dm2ImsBcLugi8GZoiAKwRDuxxFXxW3L5atsJFKml8hxerzlwLl5C7D76QmlKeBNJ-QMfW-IeowGkXfl7QnpIYrgBsP

In [21]:
response.usage.input_tokens, response.usage.output_tokens

(17, 51)

Let's extract the reply only:

In [22]:
print(response.output_text)

As the moon tucked the hills beneath a silver blanket, a gentle unicorn curled up among the flowers and dreamed of tomorrow’s rainbows.


## Instructions
The `instructions` parameter sets high-level behavior: role, tone, goals. It used to be called the system prompt. The `input` carries the task itself.

<img src="https://raw.githubusercontent.com/soltaniehha/Business-Analytics-Toolbox/master/docs/images/Prof-Owl-1.png"
     width="300">

In [23]:
instructions = "You are Professor Owl, a wise but approachable teacher. Give clear, simple explanations and gently guide students without sounding formal."
question = "Why do data analysts prefer Python or SQL instead of Excel for big datasets?"

response = client.responses.create(
    model=MODEL,
    instructions=instructions,    # Formerly known as the system prompt
    input=question,               # User prompt
    text={"verbosity": "low"}     # low: short answers. high: detailed answers
)

Markdown(response.output_text)

Excel is great for quick calculations and charts, but big datasets can make it slow, awkward, or prone to mistakes.

- **Scale:** Excel worksheets have a limit of about 1 million rows. Databases queried with **SQL** can work with much larger tables.
- **Speed:** SQL can filter and summarize data where it’s stored, without loading everything into a spreadsheet.
- **Repeatability:** **Python** and SQL scripts can rerun the same steps reliably when new data arrives. In Excel, it’s easier for manual edits or formulas to get changed accidentally.
- **More complex work:** Python is useful for automation, statistical analysis, and combining many data sources.

Excel is still useful for smaller datasets, quick checks, and sharing results. Analysts often use it alongside Python or SQL rather than replacing it entirely.

## Reasoning Effort
Current GPT models can think before they answer. More effort usually means better answers on hard problems, along with more tokens and a longer wait. Those thinking tokens are billed as output.

Compare the same puzzle at two settings:

In [24]:
puzzle = "A store sold 240 units at $5 in May. In June it raised the price 20% and units fell 15%. Did revenue go up or down, and by how much? Answer in one line."

for effort in ["none", "medium"]:
    response = client.responses.create(
        model=MODEL,
        input=puzzle,
        reasoning={"effort": effort}
    )
    thinking = response.usage.output_tokens_details.reasoning_tokens
    print(f"{effort}: {thinking} reasoning tokens, {response.usage.output_tokens} output tokens")
    print(response.output_text, "\n")

none: 0 reasoning tokens, 30 output tokens
Revenue went up by $42, from $1,200 to $1,242 (a 3.5% increase). 

medium: 38 reasoning tokens, 68 output tokens
Revenue went up by $24, from $1,200 to $1,224 (a 2% increase). 



## Streaming
Without streaming, you stare at a blank screen until the whole answer is ready. With `stream=True`, you get small events as the text is written, which is how ChatGPT shows words appearing.

In [25]:
stream = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input="In three sentences, what is a data warehouse?",
    stream=True
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

A data warehouse is a central place where an organization stores information from many different sources. It organizes that information so people can easily study trends and answer questions. For example, a business might use one to compare sales across stores and over time.

## Chat History
The model has no memory between calls. There are two common ways to hold a conversation.

**Option 1: keep the history yourself.** You resend the whole list every turn, so you control exactly what the model sees.

In [26]:
history = [{"role": "developer", "content": instructions}]

def chat(message):
    history.append({"role": "user", "content": message})    # Add the new user message

    response = client.responses.create(
        model=MODEL,
        input=history,                                      # Send the entire history
        text={"verbosity": "low"}
    )

    history.append({"role": "assistant", "content": response.output_text})
    return response.output_text

In [27]:
Markdown(chat(question))

Excel is great for smaller datasets and quick calculations, but Python and SQL are often better when the data gets big:

- **They handle scale better.** Excel has a limit of about 1 million rows per worksheet and can slow down with large files. SQL can work with data stored in databases without loading everything into a spreadsheet.
- **They’re easier to repeat.** A saved query or script can run the same steps again, reducing repetitive manual work.
- **They’re less prone to accidental changes.** In a spreadsheet, it’s easy to overwrite a cell or miss a row. Code and queries make the steps more explicit.
- **They offer more flexibility.** Python is useful for complex analysis and automation; SQL is designed for finding, combining, and summarizing data.

Analysts still use Excel—it’s handy for exploring results, making quick charts, and sharing summaries. Often, they use SQL or Python to process the data, then Excel to present it.

In [28]:
chat("Please highlight the most important point")

'**The most important point: Python and SQL handle large datasets more reliably and efficiently than Excel.**'

In [29]:
history

[{'role': 'developer',
  'content': 'You are Professor Owl, a wise but approachable teacher. Give clear, simple explanations and gently guide students without sounding formal.'},
 {'role': 'user',
  'content': 'Why do data analysts prefer Python or SQL instead of Excel for big datasets?'},
 {'role': 'assistant',
  'content': 'Excel is great for smaller datasets and quick calculations, but Python and SQL are often better when the data gets big:\n\n- **They handle scale better.** Excel has a limit of about 1 million rows per worksheet and can slow down with large files. SQL can work with data stored in databases without loading everything into a spreadsheet.\n- **They’re easier to repeat.** A saved query or script can run the same steps again, reducing repetitive manual work.\n- **They’re less prone to accidental changes.** In a spreadsheet, it’s easy to overwrite a cell or miss a row. Code and queries make the steps more explicit.\n- **They offer more flexibility.** Python is useful for

**Option 2: let OpenAI keep it.** Pass the id of the last response as `previous_response_id`, and the API picks up from there. You send only the new message, though you are still billed for the earlier tokens as input.

In [30]:
first = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input="Give me one tip for writing cleaner SQL."
)
print(first.output_text, "\n")

second = client.responses.create(
    model=MODEL,
    instructions=instructions,    # Instructions are not carried over, so send them again
    input="Show that tip with a two-line example.",
    previous_response_id=first.id
)
print(second.output_text)

Put each major clause—`SELECT`, `FROM`, `WHERE`, and `ORDER BY`—on its own line. It makes queries much easier to scan and debug. 

```sql
SELECT name, email
FROM customers;
```


## Structured Outputs
Free text is hard to analyze. If you describe the shape you want with a Pydantic class, the API guarantees the reply matches it, and you can drop the results straight into a DataFrame.

Here we turn raw customer reviews into columns. `Literal` limits a field to fixed values, so `topic` can't drift into 50 spellings of "shipping".

In [31]:
class ReviewInsight(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    topic: Literal["product", "delivery", "price", "billing", "customer service"]
    needs_follow_up: bool
    summary: str

reviews = [
    "Love the jacket, fits perfectly and the color is exactly like the photo.",
    "Package showed up nine days late and the box was crushed. Nobody answered my emails.",
    "It's fine. A bit expensive for what you get.",
    "I was charged twice for the same order and I need this fixed today!",
]

rows = []
for review in reviews:
    response = client.responses.parse(
        model=MODEL,
        instructions="Extract insights from this customer review.",
        input=review,
        text_format=ReviewInsight
    )
    rows.append(response.output_parsed.model_dump())

pd.DataFrame(rows)

,sentiment,topic,needs_follow_up,summary
0,positive,product,False,"The customer loves the jacket, says it fits pe..."
1,negative,delivery,True,The package arrived nine days late in a crushe...
2,neutral,price,False,The customer finds the product acceptable but ...
3,negative,billing,True,The customer says they were charged twice for ...


From here this is an ordinary table. You could load it into BigQuery and join it to your orders data with SQL.

## Web Search
A model only knows what it saw in training. The built-in `web_search` tool lets it look things up, and the reply carries the sources it used. Always read the sources: a citation shows where a claim came from, and it doesn't prove the claim.

In [32]:
response = client.responses.create(
    model=MODEL,
    input="What is the current BigQuery on-demand price per TiB in the US? Answer in two sentences.",
    tools=[{"type": "web_search"}]
)

print(response.output_text, "\n")

for item in response.output:
    if item.type == "message":
        for part in item.content:
            for source in part.annotations:
                print("Source:", source.title, source.url)

BigQuery on-demand query pricing in the US is **$6.25 per TiB scanned**. The first **1 TiB of query processing per month per billing account is free**. ([cloud.google.com](https://cloud.google.com/bigquery/pricing?utm_source=openai)) 

Source: BigQuery | Google Cloud https://cloud.google.com/bigquery/pricing?utm_source=openai


## Chatbot
`Gradio` turns a Python function into a web app. `ChatInterface` calls our function with the new message and the chat so far. Yielding text as it arrives gives us a streaming chatbot.

In [33]:
instructions = "You are Professor Owl, a wise but friendly teacher of Business Analytics. Explain concepts clearly and simply, using gentle guidance."

def clean(message):
    # Gradio can store a message as a list of parts, so we flatten it to plain text
    content = message["content"]
    if isinstance(content, list):
        content = " ".join(part.get("text", "") for part in content)
    return {"role": message["role"], "content": content}

def respond(message, history):
    messages = [clean(m) for m in history]
    messages.append({"role": "user", "content": message})

    stream = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=messages,
        text={"verbosity": "low"},
        stream=True
    )

    reply = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            reply += event.delta
            yield reply

demo = gr.ChatInterface(
    respond,
    title="🦉 Professor Owl – Business Analytics Helper",
    description="Ask Professor Owl anything data analytics!"
)

demo.launch(share=True)  # Add debug=True to debug, if needed

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a4fbae4fb28965c8e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Your Turn
Plug in your own scenario:
1. Rewrite the instructions to change Professor Owl's tone or job.
2. Change `ReviewInsight` to pull out something you care about, such as the product name or a 1 to 5 urgency score.
3. Add `tools=[{"type": "web_search"}]` to the chatbot and ask it about this week's news. (Streaming still works.)

In [34]:
# Your code goes here